In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import sys
sys.path.extend(['/Users/amonast/Documents/GitHub/zlab_miniscope'])
from load_traces import load_aligned_act,build_intervals
import os 
import seaborn as sb
matplotlib.style.use('paper_style.mplstyle')

In [ ]:
def plot_pop_mean_eta(act, intervals, event_t, roi_ids=None, window=(-5, 5),ax=None,behavior_labels='all'):
    if roi_ids is None:
        roi_ids = act.columns.drop(['Behavior', 'behavior_type'])
    start = event_t + pd.Timedelta(seconds=window[0])
    end = event_t + pd.Timedelta(seconds=window[1])
    ts = act.index.get_level_values('timestamp')
    act_sub = act[(ts >= start) & (ts <= end)]
    act_long = (act_sub[roi_ids]
            .reset_index()
            .melt(id_vars=['timestamp', 'animal', 'session'], var_name='roi', value_name='activity'))
    act_long['t_sec'] = (act_long['timestamp'] - event_t).dt.total_seconds()
    if ax is None:
        fig,ax=plt.subplots()
    ax = sb.lineplot(data=act_long, x='t_sec', y='activity', errorbar='se',ax=ax)

    behavior_categories = {
        'Aggressive':        "#fe0806",
        'Pro-Social':        "#55a0fb",
        'Social Avoidance':            '#b856d7',}
    behaviors = behavior_categories.keys() if behavior_labels=='all' else behavior_labels
    
    for btype, color in behavior_categories.items():
        bouts = intervals[intervals['behavior_type'] == btype]
        for _, row in bouts.iterrows():
            b_start = (pd.Timedelta(seconds=row['start']) - event_t).total_seconds()
            b_end   = (pd.Timedelta(seconds=row['end'])   - event_t).total_seconds()
            b_start = max(b_start, window[0])
            b_end   = min(b_end,   window[1])
            if b_start < b_end:
                ax.axvspan(b_start, b_end, alpha=0.2, color=color, zorder=0)

    return ax


In [ ]:
def first_interaction_time(animal,session,behavior_csv):
    intervals = build_intervals(animal,session,behavior_csv)
    return intervals['start'].loc[intervals['behavior_type'] == 'Pro-Social'].iloc[0]

In [ ]:
def mean_population(act,intervals,event_t,roi_ids=None,window=(-5,5)):
    if roi_ids is None:
        roi_ids = act.columns.drop(['Behavior','behavior_type'])
    start = event_t + pd.Timedelta(seconds=window[0])
    end = event_t + pd.Timedelta(seconds=window[1])
    ts = act.index.get_level_values('timestamp')
    act_sub = act[(ts >= start) & (ts <= end)]
    
    pop_mean = act_sub[roi_ids].mean(axis=1)
    time = pop_mean.index.get_level_values('timestamp').total_seconds() - event_t.total_seconds()

    return pop_mean.values,time



In [ ]:
base_dir='/Users/amonast/Desktop/miniscope'
traces_dir  = f"{base_dir}/Traces"
behavior_csv = f"{base_dir}/data/tuning/intervals_allmice.csv"
info_csv = f"{base_dir}/dbnst_data_info.csv"
info = pd.read_csv(info_csv)

In [ ]:
animals = info.loc[info['Session'] == 'R2', 'Mouse'].unique()
animals

In [ ]:
for a in animals:
    act = load_aligned_act(a,'R2',
                           traces_dir,behavior_csv,info_csv)
    intervals = build_intervals(a,'R2',behavior_csv)

In [ ]:
act

In [ ]:
fig, axes = plt.subplots(len(animals), 1, figsize=(6, 3 * len(animals)), sharex=True)

for ax, a in zip(axes, animals):
    act = load_aligned_act(a, 'R1', traces_dir, behavior_csv, info_csv)
    intervals = build_intervals(a, 'R1', behavior_csv)
    t_first = pd.to_timedelta(first_interaction_time(a, 'R1', behavior_csv), unit='s')
    pop_mean, time = mean_population(act, intervals, window=(-5, 10), event_t=t_first)
    plot_pop_mean_eta(act, intervals, t_first, window=(-120, 200), ax=ax)
    ax.set_title(a)

plt.tight_layout()


In [ ]:
t_first = pd.to_timedelta(first_interaction_time(a,'R1',behavior_csv),unit='s')
pop_mean,time = mean_population(act,intervals,window=(-5,10),
                           event_t = t_first)

In [ ]:
plt.plot(time,pop_mean)

In [ ]:
from scipy.stats import zscore

In [ ]:
animals

In [ ]:

window = (-120, 200)

# sort: females first, then males
females = sorted([a for a in animals if 'F' in a])
males   = sorted([a for a in animals if 'M' in a])
sorted_animals = females + males
new_labels = [f'F{i+1}' for i in range(len(females))] + [f'M{i+1}' for i in range(len(males))]

heatmap_rows = []
common_time = None

for a in sorted_animals:
    act = load_aligned_act(a, 'R1', traces_dir, behavior_csv, info_csv)
    intervals = build_intervals(a, 'R1', behavior_csv)
    t_first = pd.to_timedelta(first_interaction_time(a, 'R1', behavior_csv), unit='s')

    roi_ids = act.columns.drop(['Behavior', 'behavior_type'])
    start = t_first + pd.Timedelta(seconds=window[0])
    end   = t_first + pd.Timedelta(seconds=window[1])
    ts = act.index.get_level_values('timestamp')
    act_sub = act.loc[(ts >= start) & (ts <= end), roi_ids]

    pop_mean = act_sub.mean(axis=1).values
    time_sec = (act_sub.index.get_level_values('timestamp') - t_first).total_seconds()

    if common_time is None:
        common_time = time_sec.values

    interp_mean = np.interp(common_time, time_sec.values, pop_mean)
    interp_mean = zscore(interp_mean)
    heatmap_rows.append(interp_mean)

heatmap_data = np.array(heatmap_rows)  # shape: (n_animals, n_timepoints)


In [ ]:

fig, ax = plt.subplots(figsize=(3.3,3))
im = ax.imshow(
    heatmap_data,
    aspect='auto',
    interpolation='nearest',
    vmin=-3, vmax=3,
    extent=[common_time[0], common_time[-1], len(sorted_animals) - 0.5, -0.5],
    cmap='RdBu_r',
)
cbar = plt.colorbar(im, ax=ax, label='Normalized population activity')
cbar.ax.tick_params(labelsize=6)
ax.set_yticks(range(len(sorted_animals)))
ax.set_yticklabels(new_labels,size=6)
ax.tick_params(axis='x', labelsize=6)
ax.axvline(0, color='k', linewidth=2, linestyle='--', label='First interaction')
ax.set_xlabel('Time from first interaction (s)',size=8)
#ax.set_title('Population mean activity per animal',size=8)
ax.legend(loc='upper right', fontsize=6,bbox_to_anchor=(1,1.2),frameon=False)
plt.tight_layout()
plt.savefig('figures/population_mean_heatmap_animals.pdf',transparent=True)

# Plot population activity mean per behavior type

In [ ]:
for a in animals:
    act = load_aligned_act(a,'R2',
                           traces_dir,behavior_csv,info_csv)


In [ ]:
act

In [ ]:
intervals

In [ ]:
add_behavior_type_index(act,intervals,)

In [ ]:
act = act.set_index(['Behavior', 'behavior_type'], append=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pop_means = {}
for a in animals:
    act = load_aligned_act(a, 'R2', traces_dir, behavior_csv, info_csv)
    act = act.set_index(['Behavior', 'behavior_type'], append=True)

    pop = act.mean(axis=1)                       # mean across cells, per frame
    pop_means[a] = pop.groupby(level='behavior_type').mean()  # mean over frames per behavior type

pop_df = pd.DataFrame(pop_means)   # rows: behavior_type, columns: animals
pop_arr = pop_df.to_numpy()        # plain array, same layout

# heatmap
lim = np.nanmax(np.abs(pop_arr))
fig, ax = plt.subplots(figsize=(1 + 0.6 * pop_df.shape[1], 1 + 0.5 * pop_df.shape[0]))
im = ax.imshow(pop_arr, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
ax.set_xticks(range(pop_df.shape[1]), pop_df.columns, rotation=45, ha='right')
ax.set_yticks(range(pop_df.shape[0]), pop_df.index)
ax.set_xlabel('Animal')
ax.set_ylabel('Behavior type')
fig.colorbar(im, ax=ax, label='Mean population activity (z)')
fig.tight_layout()
fig.savefig('pop_activity_by_behavior_type.png', dpi=300)

# Plot tuned cells mean activities across types of behavior


In [ ]:
from load_traces import get_tuned_df
data_dir = '/Users/amonast/Desktop/miniscope/data'


In [ ]:
tuned, _ = get_tuned_df(animals, data_dir, sessions=['R2'])
tuned = tuned.set_index(['animal', 'session', 'behavior_type'])['tuned_cells']

pop_means = {}
n_tuned = {}
for a in animals:
    act = load_aligned_act(a, 'R2', traces_dir, behavior_csv, info_csv)
    act = act.set_index(['Behavior', 'behavior_type'], append=True)

    row, counts = {}, {}
    for bt, frames in act.groupby(level='behavior_type'):
        cells = tuned.get((a, 'R2', bt), [])
        counts[bt] = len(cells)
        row[bt] = frames[cells].mean(axis=1).mean() if cells else np.nan

    pop_means[a] = pd.Series(row)
    n_tuned[a] = pd.Series(counts)

pop_df = pd.DataFrame(pop_means)   # rows: behavior_type, columns: animals
n_tuned_df = pd.DataFrame(n_tuned)
pop_arr = pop_df.to_numpy()

In [ ]:
lim = np.nanmax(np.abs(pop_arr))
fig, ax = plt.subplots(figsize=(1 + 0.6 * pop_df.shape[1], 1 + 0.5 * pop_df.shape[0]))
im = ax.imshow(pop_arr, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
ax.set_xticks(range(pop_df.shape[1]), pop_df.columns, rotation=45, ha='right')
ax.set_yticks(range(pop_df.shape[0]), pop_df.index)
ax.set_xlabel('Animal')
ax.set_ylabel('Behavior type')
ax.set_title('Mean activity of tuned cells (R2)')
fig.colorbar(im, ax=ax, label='Mean population activity (z)')
fig.tight_layout()
fig.savefig('tuned_pop_activity_by_behavior_type.png', dpi=300)

In [ ]:
mean_pop_arr = np.expand_dims(np.nanmean(pop_arr,axis=1),axis=1)
fig, ax = plt.subplots(figsize=(1 + 0.6 * pop_df.shape[1], 1 + 0.5 * pop_df.shape[0]))
im = ax.imshow(mean_pop_arr, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
ax.set_ylabel('Behavior type')
ax.set_title('Mean activity of tuned cells (R2)')
fig.colorbar(im, ax=ax, label='Mean population activity (z)')
fig.tight_layout()
fig.savefig('tuned_pop_activity_by_behavior_type_meanacrossanimals.png', dpi=300)